# Mini Project 1 — E-commerce Refund Triage with Azure AI Foundry Agent

This industry-style mini project uses the deployed Azure AI Foundry hosted agent **`sandeepagent11`** to assess a refund request, recommend a policy-aligned action, and draft a customer reply.

The endpoint and API key stay in `.env`; they are never written into this notebook.

In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

AGENT_ENDPOINT = os.environ['AGENT_ENDPOINT']
API_KEY = os.environ['AZURE_OPENAI_API_KEY']
API_VERSION = 'v1'

print('Configured agent endpoint:', AGENT_ENDPOINT.split('/agents/')[0] + '/agents/sandeepagent11/...')

In [2]:
def get_output_text(response_json: dict) -> str:
    """Extract assistant text from an OpenAI Responses API result."""
    for item in response_json.get('output', []):
        if item.get('type') == 'message':
            for part in item.get('content', []):
                if part.get('type') == 'output_text':
                    return part['text']
    raise ValueError(f'No assistant text found: {response_json}')


def ask_refund_agent(ticket: str) -> str:
    prompt = f'''You are a customer-support operations agent for an e-commerce retailer.
Analyze the refund request below and respond with these exact headings:
Summary, Risk level, Recommended next action, Customer reply draft, Human approval required.
Apply the stated policy. Do not invent customer or order details.

{ticket}'''

    response = requests.post(
        AGENT_ENDPOINT,
        params={'api-version': API_VERSION},
        headers={'api-key': API_KEY, 'Content-Type': 'application/json'},
        json={'input': prompt, 'stream': False},
        timeout=120,
    )
    response.raise_for_status()
    return get_output_text(response.json())

In [3]:
sample_ticket = '''Ticket: Customer Priya says order ORD-10452 was delivered two days late.
She paid ₹4,999 for a premium plan and requests a full refund.
The stated service policy offers a 10% service-credit for delivery delays,
while full refunds require a damaged or missing order. No damage has been reported.'''.strip()

agent_result = ask_refund_agent(sample_ticket)
print(agent_result)

Summary
- Customer Priya reports order ORD-10452 arrived two days late and requests a full refund of the premium plan fee (4,999). Current policy grants a 10% service credit for late delivery; full refunds apply only to damaged or missing orders. No damage/missing reported.

Risk level
Low

Recommended next action
- Verify shipment timeline for ORD-10452 (promised vs. actual delivery dates) to confirm the two-day delay.
- Confirm no damage/missing-item reports on the order.
- Apply the policy: offer a 10% service credit on the premium plan fee (amount: 499.90 in the original payment currency) to the customer’s account.
- Decline the full refund request, citing the delivery-delay policy and eligibility for service credit instead.

Customer reply draft
Hi Priya, I’m sorry your order arrived two days later than expected. Under our service policy, delivery delays are eligible for a 10% service credit, while full refunds are reserved for orders that are damaged or missing. I can apply a 10%

## Production notes

- Keep policy documents and order data in approved enterprise systems; send only the minimum necessary data to the agent.
- Keep human approval for exceptions, large refunds, and policy overrides.
- Log the ticket ID, recommendation, approver, and executed action for auditability.
- Use `previous_response_id` and `agent_session_id` if the workflow needs multi-turn follow-up with the same hosted-agent session.